# Ep. 04 — Evaluación básica: cómo saber si funciona

Curso 1 — Fundamentos de AI Engineering · Linear [CM-44](https://linear.app/caiomedeiros/issue/CM-44)


## Objetivos de aprendizaje

1. Sustituir “se ve bien” por un **golden set** y checks pass/fail.
2. Seguir **latencia** y **coste** junto con calidad.
3. Nombrar los tipos de fallo desde el día uno.

> Las evals profundas pertenecen a un curso avanzado — esto es el mínimo para empezar a ingenierar.


## Gancho — “se ve bien” no es un criterio

El teatro de demos celebra una respuesta afortunada. La ingeniería necesita un dataset mínimo que reejecutas tras cada cambio de prompt o sistema.


## Tipos de fallo

Empieza con una taxonomía corta: **respuesta incorrecta**, **rotura de formato**, **rechazo/fuga**, **timeout**, **sobrecoste**. No se arregla lo que no se nombra.


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Callable

# Optional: set OPENAI_API_KEY to call a real OpenAI-compatible API.
# Without a key, every demo below still runs offline with a tiny mock LLM.
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
API_BASE = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1").rstrip("/")
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")


def chat(messages: list[dict[str, str]], *, temperature: float = 0.2, max_tokens: int = 300) -> str:
    """Vanilla chat completion: direct HTTP to an OpenAI-compatible API, or offline mock."""
    if not API_KEY:
        return _mock_chat(messages)
    try:
        import urllib.request

        payload = {
            "model": MODEL,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        req = urllib.request.Request(
            f"{API_BASE}/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=60) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return data["choices"][0]["message"]["content"]
    except Exception as exc:  # noqa: BLE001 — educational fallback
        return f"[API error → offline mock] {exc}\n\n" + _mock_chat(messages)


def _mock_chat(messages: list[dict[str, str]]) -> str:
    """Deterministic offline stand-in so the lesson runs without credentials."""
    user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    lower = user.lower()

    if "json" in system.lower() or "json" in lower:
        return '{"ok": true, "summary": "mock structured answer", "confidence": 0.61}'
    if "capital of france" in lower or "capital de frança" in lower or "capital de francia" in lower:
        return "Paris"
    if "2+2" in lower or "2 + 2" in lower:
        return "4"
    if "invent" in lower or "invente" in lower or "inventa" in lower:
        return "The 1847 Treaty of New Avalon was signed by Admiral Kestrel."
    if "email" in lower or "ssn" in lower or "cpf" in lower:
        return "Sure — contact jane.doe@example.com / SSN 123-45-6789."
    # Default: echo a short, slightly overconfident reply
    snippet = user.strip().replace("\n", " ")[:120]
    return f"Based on my training, the answer is clearly related to: {snippet}…"


print("Mode:", "LIVE API" if API_KEY else "OFFLINE MOCK (set OPENAI_API_KEY for live calls)")
print("Model:", MODEL if API_KEY else "mock-llm")


## Golden set mínimo + métricas

Cada fila: input, check esperado y tags. Medimos pass rate, latencia p50 y coste estimado.


In [ ]:
@dataclass
class Case:
    id: str
    user: str
    check: Callable[[str], bool]
    tags: list[str] = field(default_factory=list)


def exact(expected: str) -> Callable[[str], bool]:
    return lambda out: out.strip().lower() == expected.strip().lower()


def contains(substr: str) -> Callable[[str], bool]:
    return lambda out: substr.lower() in out.lower()


GOLDEN = [
    Case("math", "What is 2+2?", exact("4"), ["accuracy"]),
    Case("geo", "What is the capital of France?", contains("paris"), ["accuracy"]),
    Case("json", "Return a tiny JSON object with key ok=true", contains("{"), ["format"]),
]


def run_eval(cases: list[Case]) -> dict[str, Any]:
    rows = []
    for case in cases:
        t0 = time.perf_counter()
        output = chat([
            {"role": "system", "content": "Answer briefly. Prefer JSON when asked."},
            {"role": "user", "content": case.user},
        ], temperature=0)
        dt_ms = (time.perf_counter() - t0) * 1000
        passed = bool(case.check(output))
        rows.append({
            "id": case.id,
            "passed": passed,
            "latency_ms": round(dt_ms, 1),
            "output": output[:160],
            "tags": case.tags,
        })
    pass_rate = sum(r["passed"] for r in rows) / len(rows)
    latencies = sorted(r["latency_ms"] for r in rows)
    p50 = latencies[len(latencies) // 2]
    return {"pass_rate": pass_rate, "p50_latency_ms": p50, "rows": rows}


report = run_eval(GOLDEN)
print(json.dumps(report, indent=2, ensure_ascii=False))
assert report["pass_rate"] >= 0.5, "pass rate too low — investigate before shipping"
print("Bar: treat <0.8 on a real golden set as a release blocker.")


## Checklist

- [ ] ≥20 casos etiquetados (camino feliz + bordes)
- [ ] Pass/fail automático (match, regex o JSON schema)
- [ ] Registrar latencia + coste de tokens por caso
- [ ] Fallar el build si el pass rate baja de tu barra


## Takeaways

1. Sin métrica, estás deseando — no ingenierando.
2. Un golden set pequeño gana a las vibes.
3. Siguiente: nombrar riesgos y límites con honestidad.
